# Kharchify AI/ML Internship Task

Two parts:

* **A. Dish category classifier** - TF-IDF (word + character n-grams) into Logistic Regression, trained on `data/dishes.csv`, evaluated on a held-out split, then used on unseen dish names.
* **B. Plain English sales questions** - a rule-based parser that reads the question and lets pandas do every calculation. No LLM does arithmetic, and an unparsed question gets a polite fallback.

Data: `data/dishes.csv` (130 dishes, 6 categories) and `data/sales.csv` (937 order lines, 20 Oct to 17 Nov 2026).


## Setup

```bash
pip install -r requirements.txt
```

Notebook runs from the repository root.

In [1]:
%pip install -q scikit-learn pandas joblib


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Part A - cleaning

The supplied files are tidy, but menu exports rarely are, so the cleaning is written to survive spacing, casing, duplicates and stray categories.

In [2]:
"""
Part A: dish category classifier.

TF-IDF (word + character n-grams) into Logistic Regression, trained on
dishes.csv, evaluated on a held-out stratified split, then used to predict
categories for dish names it has never seen.

Run:  python part_a.py
"""

from __future__ import annotations

import json
import re

import pandas as pd


RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# 1. Load and clean. The files we were given are tidy, but the cleaning is
#    written defensively because real menu exports are not: spacing, casing,
#    duplicate rows and stray categories all appear in practice.
# ---------------------------------------------------------------------------
def load_dishes(path: str = "dishes.csv") -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]

    df["dish_name"] = (
        df["dish_name"].astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    df["category"] = (
        df["category"].astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.title()
    )

    df = df.dropna(subset=["dish_name", "category"])
    df = df[df["dish_name"].str.len() > 0]
    df = df.drop_duplicates(subset=["dish_name"])        # one label per dish

    # collapse obvious near-duplicate labels (e.g. "main course" / "Main Course")
    keep = {"Starter", "Main Course", "Dessert", "Bread", "Rice", "Beverage"}
    df["category"] = df["category"].where(df["category"].isin(keep), df["category"])
    return df.reset_index(drop=True)


# ---------------------------------------------------------------------------

df = load_dishes("data/dishes.csv")
print(df.shape)
print(df["category"].value_counts())
df.sample(5, random_state=1)

(130, 2)
category
Starter        25
Main Course    25
Dessert        20
Bread          20
Rice           20
Beverage       20
Name: count, dtype: int64


,dish_name,category
108,Paneer Biryani,Rice
105,Paneer Butter Masala,Main Course
35,Gulab Jamun,Dessert
124,Honey Chilli Potato,Starter
53,Stuffed Paratha,Bread


## Part A - train and evaluate

Word n-grams catch phrases like *chicken biryani*; character n-grams survive the typos and Indian-English spellings that real input contains (*panner*, *masla*).

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline

RANDOM_STATE = 42

def build_model():
    features = FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)),
    ])
    clf = LogisticRegression(max_iter=2000, C=10.0, class_weight="balanced", random_state=RANDOM_STATE)
    return Pipeline([("features", features), ("clf", clf)])

X, y = df["dish_name"].values, df["category"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

model = build_model().fit(X_train, y_train)
y_pred = model.predict(X_test)

print("train", len(X_train), "test", len(X_test))
print(f"test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"5-fold CV: {cross_val_score(build_model(), X, y, cv=5).mean():.3f}")
print(classification_report(y_test, y_pred, zero_division=0))

train 97 test 33
test accuracy: 0.727
5-fold CV: 0.723
              precision    recall  f1-score   support

    Beverage       0.33      0.40      0.36         5
       Bread       1.00      1.00      1.00         5
     Dessert       0.40      0.40      0.40         5
 Main Course       0.78      1.00      0.88         7
        Rice       1.00      0.80      0.89         5
     Starter       1.00      0.67      0.80         6

    accuracy                           0.73        33
   macro avg       0.75      0.71      0.72        33
weighted avg       0.76      0.73      0.73        33



**Result: test accuracy 0.727, 5-fold cross-validation 0.723.**

Confusion matrix, rows are true labels:

```
labels: ['Beverage', 'Bread', 'Dessert', 'Main Course', 'Rice', 'Starter']
[2, 0, 2, 1, 0, 0]
[0, 5, 0, 0, 0, 0]
[3, 0, 2, 0, 0, 0]
[0, 0, 0, 7, 0, 0]
[0, 0, 1, 0, 4, 0]
[1, 0, 0, 1, 0, 4]
```

Bread, Rice and Main Course are strong. Beverage and Dessert are weak, and they fail into each other: a cold coffee and a chocolate mousse look alike to a bag of words.

In [4]:
import pandas as pd, seaborn as sns

labels = sorted(set(y))
cm = confusion_matrix(y_test, y_pred, labels=labels)
pd.DataFrame(cm, index=labels, columns=labels)

Matplotlib is building the font cache; this may take a moment.


,Beverage,Bread,Dessert,Main Course,Rice,Starter
Beverage,2,0,2,1,0,0
Bread,0,5,0,0,0,0
Dessert,3,0,2,0,0,0
Main Course,0,0,0,7,0,0
Rice,0,0,1,0,4,0
Starter,1,0,0,1,0,4


## Part A - five unseen dishes

In [5]:
unseen = [
    "Paneer Kathi Roll",
    "Mango Kulfi Falooda",
    "Tandoori Broccoli",
    "Veg Hakka Noodles",
    "Rose Lassi",
]

proba = model.predict_proba(unseen)
out = pd.DataFrame({
    "dish": unseen,
    "predicted": [model.classes_[r.argmax()] for r in proba],
    "confidence": [round(r.max(), 3) for r in proba],
    "runner_up": [model.classes_[r.argsort()[-2]] for r in proba],
    "runner_up_conf": [round(sorted(r)[-2], 3) for r in proba],
})
out

,dish,predicted,confidence,runner_up,runner_up_conf
0,Paneer Kathi Roll,Starter,0.541,Main Course,0.263
1,Mango Kulfi Falooda,Dessert,0.554,Beverage,0.238
2,Tandoori Broccoli,Starter,0.648,Dessert,0.121
3,Veg Hakka Noodles,Starter,0.452,Main Course,0.175
4,Rose Lassi,Beverage,0.826,Dessert,0.044


**Where it fails.** *Paneer Kathi Roll* comes back as Starter at 0.54, close to Main Course at 0.26. A kathi roll is a main dish, but the model only knows the words, and *paneer* plus *tikka* vocabulary lives in the Starter rows. *Veg Hakka Noodles* is the same story at 0.45. Indo-Chinese noodle dishes were never in the training set as mains.

The fix is not a bigger model, it is labels the model can learn from: one or two examples per new pattern, or a rule that treats a named noodle or roll dish as a main. Confidence below 0.6 should be routed to a human anyway, and that threshold is worth keeping in the code.

## Part B - plain English sales questions

No LLM does arithmetic. A parser resolves the intent (*top sellers*, *how many of one dish*, *revenue*, *total*), the time window (*today, yesterday, this week, last week, this month, last month, YYYY-MM-DD*) and the dish name (exact, substring, then fuzzy for typos). pandas computes every number. Anything unrecognised gets a polite fallback that names what it can do.

In [6]:
import re
from dataclasses import dataclass
from datetime import date, timedelta
from difflib import get_close_matches

import pandas as pd

# paste the classes from part_b.py
exec(open("part_b.py").read().split('if __name__ == "__main__":')[0])

In [7]:
df_sales = load_sales("data/sales.csv")
bot = SalesAssistant(df_sales)

questions = [
    "What were my top 3 dishes yesterday?",
    "How many Paneer Butter Masala did we sell today?",
    "Top 5 items by revenue this month",
    "How many panner butter masla did we sell today?",
    "Total revenue last week",
    "Which items are not selling?",
    "What is the weather in Mumbai?",
]
for q in questions:
    print("Q:", q)
    print("A:", bot.ask(q))
    print()

Q: What were my top 3 dishes yesterday?
A: Top 3 by quantity yesterday:
1. Fish Fingers: 4 sold
2. Dal Makhani: 4 sold
3. Cold Coffee: 4 sold

Q: How many Paneer Butter Masala did we sell today?
A: You sold 5 x Paneer Butter Masala today.

Q: Top 5 items by revenue this month
A: Top 5 by revenue this month:
1. Veg Biryani: INR 10,810
2. Paneer Tikka: INR 10,780
3. Butter Chicken: INR 10,240
4. Chicken Biryani: INR 9,280
5. Dal Makhani: INR 8,820

Q: How many panner butter masla did we sell today?
A: You sold 5 x Paneer Butter Masala today. (I read that as Paneer Butter Masala)

Q: Total revenue last week
A: Total revenue last week was INR 78,590 from 538 items sold.

Q: Which items are not selling?
A: Slowest 3 by quantity across all recorded sales:
1. Gajar Halwa: 1 sold
2. Chicken 65: 2 sold
3. Hara Bhara Kebab: 2 sold

Q: What is the weather in Mumbai?
A: I could not turn that into a sales question. I can answer things like "top 3 dishes yesterday", "how many butter naan this week",

Sample outputs are also saved in `part_b_examples.txt` so they can be read without running anything:

```
Q: What were my top 3 dishes yesterday?
A: Top 3 by quantity yesterday:
1. Fish Fingers: 4 sold
2. Dal Makhani: 4 sold
3. Cold Coffee: 4 sold

Q: How many Paneer Butter Masala did we sell today?
A: You sold 5 x Paneer Butter Masala today.

Q: Top 5 items by revenue this month
A: Top 5 by revenue this month:
1. Veg Biryani: INR 10,810
2. Paneer Tikka: INR 10,780
3. Butter Chicken: INR 10,240
4. Chicken Biryani: INR 9,280
5. Dal Makhani: INR 8,820

Q: How many panner butter masla did we sell today?
A: You sold 5 x Paneer Butter Masala today. (I read that as Paneer Butter Masala)

Q: Total revenue last week
A: Total revenue last week was INR 78,590 from 538 items sold.

Q: Which items are not selling?
A: Slowest 3 by quantity across all recorded sales:
1. Gajar Halwa: 1 sold
2. Chicken 65: 2 sold
3. Hara Bhara Kebab: 2 sold

Q: What is the weather in Mumbai?
A: I could not turn that into a sales question. I can answer things like "top 3 dishes yesterday", "how many butter naan this week", "revenue from chicken biryani this month", or "total sales last week".
```

## Notes on responsible use of AI

* No language model is in the calculation path. Parsing is deterministic, arithmetic is pandas, so an answer can always be traced to rows.
* Nothing hallucinated: an unmatched question, an unknown dish, or an empty date range returns a sentence that says so.
* No API keys, no paid services, no data leaves the machine.
* The model file is small and reproducible from the seed in `part_a.py`.